In [ ]:
pip install tf-keras

In [ ]:
import os
# Force TensorFlow to use the stable Keras 2 engine
os.environ["TF_USE_LEGACY_KERAS"] = "1"

import tensorflow as tf
import tensorflow_recommenders as tfrs

print(f"TensorFlow Version: {tf.__version__}")
print("Environment successfully configured!")

Data Ingestion

pip install tensorflow-datasets


In [ ]:
pip install importlib-resources

In [ ]:
import tensorflow_datasets as tfds
import tensorflow as tf
ratings=tfds.load("movielens/100k-ratings", split="train")
movies=tfds.load("movielens/100k-movies", split="train")
print("Datasets loaded successfully!")

In [ ]:
print("--- RAW RATING DATA ---")
for rating in ratings.take(4):
    print(rating)

print("\n--- RAW MOVIE DATA ---")
for movie in movies.take(4):
    print(movie)

Data Preprocessing and Feature Extraction

In [ ]:
import numpy as np

#Pre-Processing
ratings=ratings.map(lambda x:{"movie_title": x["movie_title"], "user_id":x["user_id"]})
movies=movies.map(lambda x: x["movie_title"])

#Feature Extraction
user_ids_vocabulary = np.unique(np.concatenate(list(ratings.batch(1000).map(lambda x: x["user_id"]))))
movie_titles_vocabulary = np.unique(np.concatenate(list(movies.batch(1000))))

print(f"Total Unique Users: {len(user_ids_vocabulary)}")
print(f"Total Unique Movies: {len(movie_titles_vocabulary)}")


Defining the Task and Compiling the Model

In [ ]:


import tensorflow as tf
import tensorflow_recommenders as tfrs

embedding_dimension = 32

# --- 1. BUILD THE TOWERS ---
user_model = tf.keras.Sequential([
    tf.keras.layers.StringLookup(vocabulary=user_ids_vocabulary, mask_token=None),
    tf.keras.layers.Embedding(len(user_ids_vocabulary) + 1, embedding_dimension),
    # tf.keras.layers.Dense(32, activation="relu")
])

movie_model = tf.keras.Sequential([
    tf.keras.layers.StringLookup(vocabulary=movie_titles_vocabulary, mask_token=None),
    tf.keras.layers.Embedding(len(movie_titles_vocabulary) + 1, embedding_dimension),
    # tf.keras.layers.Dense(32, activation="relu")
])


# --- 2. DEFINE METRICS AND TASK ---
# The model will evaluate itself against the entire catalog of 1664 movies
metrics = tfrs.metrics.FactorizedTopK(
  candidates=movies.batch(128).map(movie_model)
)

task = tfrs.tasks.Retrieval(metrics=metrics)


# --- 3. ASSEMBLE THE FULL MODEL ---
class RecommenderModel(tfrs.Model):
    def __init__(self, user_model, movie_model):
        super().__init__()
        self.movie_model = movie_model
        self.user_model = user_model
        self.task = task

    def compute_loss(self, features, training=False):
        # Feed the raw IDs into the towers to get the mathematical vectors
        user_embeddings = self.user_model(features["user_id"])
        positive_movie_embeddings = self.movie_model(features["movie_title"])

        # The task calculates the distance between them and returns the Loss
        return self.task(user_embeddings, positive_movie_embeddings)


# --- 4. COMPILE ---
model = RecommenderModel(user_model, movie_model)
model.compile(optimizer=tf.keras.optimizers.Adagrad(learning_rate=0.1))

print("Model architecture compiled and ready for training!")

In [ ]:
pip install scikit-learn matplotlib

In [ ]:
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

# 1. Extract the raw 32D embeddings directly from the Movie Tower's embedding layer
# (Layer 0 is the StringLookup, Layer 1 is the actual Embedding layer)
movie_embeddings_32d = movie_model.layers[1].get_weights()[0]

# 2. Squash the 32 dimensions down to 2 dimensions using PCA
pca = PCA(n_components=2)
movie_embeddings_2d = pca.fit_transform(movie_embeddings_32d)

# 3. Plot the results
plt.figure(figsize=(10, 8))
plt.scatter(movie_embeddings_2d[:, 0], movie_embeddings_2d[:, 1], alpha=0.5, color='blue')
plt.title("Movie Embeddings (Pre-Training)")
plt.xlabel("PCA Dimension 1")
plt.ylabel("PCA Dimension 2")
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

In [ ]:
# --- PHASE 5: MODEL TRAINING ---

# 1. Prepare the Data Pipeline
# We know the dataset has exactly 100,000 ratings. 
# We shuffle them and split them: 80,000 for training, 20,000 for testing.
tf.random.set_seed(42)
shuffled_ratings = ratings.shuffle(100_000, seed=42, reshuffle_each_iteration=False)

# Create the Training set (Batched for speed)
train_dataset = shuffled_ratings.take(80_000).batch(8192).cache()

# Create the Testing set
test_dataset = shuffled_ratings.skip(80_000).take(20_000).batch(8192).cache()

# 2. Train the Model!
print("Starting training...")
# The .fit() command is what actually triggers the Adagrad optimizer to update the embeddings
history = model.fit(train_dataset, epochs=3)

# 3. Evaluate the Model
print("\nEvaluating on unseen test data...")
metrics = model.evaluate(test_dataset, return_dict=True)

# Print the final grade: How often is the correct movie in the top 100 guesses?
top_100_accuracy = metrics['factorized_top_k/top_100_categorical_accuracy']
print(f"\nFinal Top-100 Accuracy: {top_100_accuracy * 100:.2f}%")

In [ ]:
import numpy as np

# 1. Set up the Retrieval index
# This creates a super-fast search engine for our movie vectors
index = tfrs.layers.factorized_top_k.BruteForce(model.user_model)
index.subplots = False 
index.index_from_dataset(
  tf.data.Dataset.zip((movies.batch(100), movies.batch(100).map(model.movie_model)))
)

# 2. Make a Prediction!
# Let's ask the model for the Top 5 movie recommendations for User "42"
user_id = np.array(["50"])
_, titles = index(user_id)

print(f"Top 5 Recommendations for User {user_id[0]}:")
for i, title in enumerate(titles[0, :5].numpy()):
    print(f"{i+1}. {title.decode('utf-8')}")

In [ ]:
import os

# 1. Define the folder name where the model will be saved
export_path = os.path.join(".", "saved_recommender_model")

# 2. Save the BruteForce index to disk
tf.saved_model.save(index, export_path)

print(f"Success! Your model's brain has been exported to: {export_path}")

In [ ]:
# --- SIMULATING A REAL-WORLD WEB SERVER ---

print("Booting up server and loading AI...")
# Load the model directly from the hard drive
loaded_production_model = tf.saved_model.load(export_path)
print("AI Loaded!")

# An API request comes in for User "42"
incoming_user_id = np.array(["50"])

# Generate the prediction using the LOADED model
_, titles = loaded_production_model(incoming_user_id)

print(f"\nProduction Server Recommendations for User {incoming_user_id[0]}:")
for i, title in enumerate(titles[0, :5].numpy()):
    print(f"{i+1}. {title.decode('utf-8')}")